In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/09 19:13:32 WARN Utils: Your hostname, PradyunPC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/09 19:13:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 19:13:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/09 19:13:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
df_green = spark.read.option(
    "recursiveFileLookup", "true"
).parquet("/home/sujatha/projects/batching/data/pq/green/")

In [4]:
df_yellow = spark.read.option(
    "recursiveFileLookup", "true"
).parquet("/home/sujatha/projects/batching/data/pq/yellow/")

In [6]:
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

In [7]:
df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [8]:
df_yellow.show()

+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|    pickup_datetime|   dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       2|2020-01-21 11:01:28|2020-01-21 11:18:34|              1|         3.01|         1|                 N|         137|         232|           1|       14.5|  0.0|    0.5|       3.0|         0.0|                  0.3|        20.

In [9]:
df_green.show()

+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|    pickup_datetime|   dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2|2020-01-16 23:31:44|2020-01-16 23:46:21|                 N|         1|          65|          56|              1|          8.6|       24.5|  0.5|    0.5|       0.0

In [14]:
common_columns = []

df_green_columns  = set(df_green.columns)
for column in df_green_columns:
    if column in set(df_yellow.columns):
        common_columns.extend([column])
print(common_columns)

['passenger_count', 'tip_amount', 'improvement_surcharge', 'store_and_fwd_flag', 'DOLocationID', 'trip_distance', 'fare_amount', 'payment_type', 'dropoff_datetime', 'VendorID', 'mta_tax', 'tolls_amount', 'RatecodeID', 'extra', 'total_amount', 'PULocationID', 'pickup_datetime', 'congestion_surcharge']


In [18]:
from pyspark.sql import functions as F

In [19]:
df_green_sel = df_green \
    .select(common_columns) \
    .withColumn('service_type', F.lit('green'))

In [20]:
df_yellow_sel = df_yellow \
    .select(common_columns) \
    .withColumn('service_type', F.lit('yellow'))

In [21]:
trips_data = df_green_sel.unionAll(df_yellow_sel)

In [22]:
trips_data.groupBy('service_type').count().show()

[Stage 4:============================================>            (33 + 9) / 42]

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [24]:
trips_data.columns

['passenger_count',
 'tip_amount',
 'improvement_surcharge',
 'store_and_fwd_flag',
 'DOLocationID',
 'trip_distance',
 'fare_amount',
 'payment_type',
 'dropoff_datetime',
 'VendorID',
 'mta_tax',
 'tolls_amount',
 'RatecodeID',
 'extra',
 'total_amount',
 'PULocationID',
 'pickup_datetime',
 'congestion_surcharge',
 'service_type']

In [26]:
trips_data.createOrReplaceTempView('trips_data')

In [27]:
spark.sql("""
SELECT
    service_type,
    count(1)
FROM
    trips_data
GROUP BY 
    service_type
""").show()

+------------+--------+
|service_type|count(1)|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+

